# Crypto Lab 05: Hash Functions

**CYBR 3570 - Applied Cryptography**  
**Week 5: Hash Functions**

## Today's Big Question

**How can a short digest represent a large message securely enough that we can use it for integrity, signatures, and identification?**

Hash functions are used throughout modern security systems: file integrity checks, Git, digital signatures, malware detection, forensic workflows, key derivation, password storage, and protocol design.

In this lab, you will use Python to explore what cryptographic hash functions do, what security properties they are supposed to provide, and how those properties can fail in simplified settings.


## Learning Objectives

By the end of this lab, you should be able to:

1. Compute cryptographic hashes using Python.
2. Explain the difference between encryption and hashing.
3. Define preimage resistance, second-preimage resistance, and collision resistance.
4. Explain why collision attacks are faster than preimage attacks.
5. Demonstrate the avalanche effect experimentally.
6. Perform toy preimage and collision searches using truncated hashes.
7. Explain why raw hash functions should not be used as password hashing or authentication mechanisms.
8. Add basic hashing utilities to your cryptographic toolkit.


## Part 1: Setup

Run the following cell first. We will use only Python's standard library plus `matplotlib` for one optional visualization.


In [1]:
import hashlib
import hmac
import math
import os
import secrets
import statistics
import time
from collections import Counter
from pathlib import Path

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

print("Setup complete.")


Setup complete.


## Part 2: Concept Check

Answer briefly in Markdown.

1. What is the difference between hashing and encryption?
2. Why might a digital signature scheme sign a hash of a message instead of the full message?
3. If collisions are guaranteed to exist, what does it mean for a hash function to be collision resistant?


### Your Answers

1. Encryption is two way and hashing is one way

2. Space savings for hash, hash is as good as the original file

3. Wide possible values, hard to find collision, small changes in input give big changes in hash


## Part 3: Your First Hashes

Python's `hashlib` module gives access to several cryptographic hash functions.

A hash function is deterministic: the same input always produces the same digest.


In [2]:
message = b"CYBR 3570"

sha256_digest = hashlib.sha256(message).hexdigest()
sha3_digest = hashlib.sha3_256(message).hexdigest()
blake2_digest = hashlib.blake2b(message, digest_size=32).hexdigest()

print("message:", message)
print("SHA-256: ", sha256_digest)
print("SHA3-256:", sha3_digest)
print("BLAKE2b: ", blake2_digest)


message: b'CYBR 3570'
SHA-256:  0a3972483d8a5aee6352162bc6c77b71afc4a8a4a6db7c6a0637e847e64fe3ba
SHA3-256: f076ba4bc18c0f6868291592a6cd4c55069ea916f726efda191b8d20e9de7664
BLAKE2b:  ebeddd5d1496c64811bc64e80a563d39c49798745887b57701870340a0bcb87e


### Exercise 1

Write a function called `hash_message()` that accepts a message as `bytes` and an algorithm name. It should support:

- `sha256`
- `sha3_256`
- `blake2b`

Return the digest as a hexadecimal string.


In [4]:
def hash_message(message: bytes, algorithm: str = "sha256") -> str:
    """
    Return the hexadecimal digest of a message using the selected algorithm.

    Supported algorithms: sha256, sha3_256, blake2b.
    """
    # TODO: Implement this function.
    if algorithm == "sha256":
        return hashlib.sha256(message).hexdigest()
    elif algorithm == "sha3_256":
        return hashlib.sha3_256(message).hexdigest()
    elif algorithm == "blake2b":
        return hashlib.blake2b(message, digest_size=32).hexdigest()
    return hash 
    raise NotImplementedError("Complete hash_message()")


# Quick checks. These should run after your implementation.
assert hash_message(b"abc", "sha256") == hashlib.sha256(b"abc").hexdigest()
assert hash_message(b"abc", "sha3_256") == hashlib.sha3_256(b"abc").hexdigest()
assert hash_message(b"abc", "blake2b") == hashlib.blake2b(b"abc", digest_size=32).hexdigest()


## Part 4: Avalanche Effect

A good hash function should produce outputs that look unrelated even when inputs differ only slightly.

We will flip a single bit of a message and compare the resulting digest.


In [5]:
def bytes_to_bits(data: bytes) -> str:
    return "".join(f"{byte:08b}" for byte in data)


def bit_difference(a: bytes, b: bytes) -> int:
    """Count how many bit positions differ between two byte strings of equal length."""
    if len(a) != len(b):
        raise ValueError("Inputs must have equal length")
    return sum((x ^ y).bit_count() for x, y in zip(a, b))


def flip_one_bit(data: bytes, byte_index: int = 0, bit_index: int = 0) -> bytes:
    """Return a copy of data with one selected bit flipped."""
    if not (0 <= byte_index < len(data)):
        raise IndexError("byte_index out of range")
    if not (0 <= bit_index < 8):
        raise IndexError("bit_index must be between 0 and 7")
    mutable = bytearray(data)
    mutable[byte_index] ^= (1 << bit_index)
    return bytes(mutable)

original = b"Hash functions are sensitive to tiny input changes."
modified = flip_one_bit(original, byte_index=0, bit_index=0)

h1 = hashlib.sha256(original).digest()
h2 = hashlib.sha256(modified).digest()

print("Original:", original)
print("Modified:", modified)
print("SHA-256(original):", h1.hex())
print("SHA-256(modified):", h2.hex())
print("Different digest bits:", bit_difference(h1, h2), "out of", len(h1) * 8)


Original: b'Hash functions are sensitive to tiny input changes.'
Modified: b'Iash functions are sensitive to tiny input changes.'
SHA-256(original): 521373583d2c962af8b262fbbbf0e3117319a186090338bf97316748cc44b98b
SHA-256(modified): 3e3a8cffc1fd5ada98f45666cf2988c9feb42ed1c438557f433d056320bcb87f
Different digest bits: 134 out of 256


### Exercise 2

Run an avalanche experiment.

Generate 100 random 32-byte messages. For each message:

1. Flip one bit.
2. Hash both versions with SHA-256.
3. Count how many digest bits changed.

Compute the minimum, maximum, and average number of changed bits.

For a 256-bit digest, what average would you expect if each output bit changed independently with probability 1/2?


In [11]:
def avalanche_trials(trials: int = 100, message_size: int = 32) -> list[int]:
    """Run SHA-256 avalanche trials and return changed-bit counts."""
    # TODO: Implement the experiment.
    counts = []
    for count in range(trials):
        msg = secrets.token_bytes(message_size)
        msg2 = flip_one_bit(msg)
        hash1 = hashlib.sha256(msg).digest()
        hash2 = hashlib.sha256(msg2).digest()
        #print(msg)
        #print(msg2)
        diff = bit_difference(hash1, hash2)
        counts.append(diff)
    return counts
    #raise NotImplementedError("Complete avalanche_trials()")


counts = avalanche_trials(100, 32)
print("min:", min(counts))
print("max:", max(counts))
print("average:", statistics.mean(counts))
print("expected average:", 256 / 2)


min: 112
max: 147
average: 129.53
expected average: 128.0


In [ ]:
# Optional visualization after completing avalanche_trials().
# if MATPLOTLIB_AVAILABLE:
#     plt.figure()
#     plt.hist(counts, bins=15)
#     plt.xlabel("Changed digest bits")
#     plt.ylabel("Frequency")
#     plt.title("SHA-256 Avalanche Experiment")
#     plt.show()


## Part 5: Toy Preimage Search

Real hash functions are designed so preimage search is infeasible. But we can make a toy version by truncating a digest to a small number of bits.

For example, instead of using all 256 bits of SHA-256, we might keep only the first 16 bits. That makes attacks possible in class.


In [12]:
def truncated_sha256(data: bytes, bits: int) -> int:
    """
    Return the first `bits` bits of SHA-256(data) as an integer.

    This is intentionally insecure and used only for experiments.
    """
    if bits <= 0 or bits > 256:
        raise ValueError("bits must be between 1 and 256")
    digest_int = int.from_bytes(hashlib.sha256(data).digest(), "big")
    return digest_int >> (256 - bits)

print(truncated_sha256(b"abc", 16))
print(hex(truncated_sha256(b"abc", 16)))


47736
0xba78


### Exercise 3

Write a function `find_preimage()` that searches for a random message whose truncated SHA-256 digest equals a target value.

Use `secrets.token_bytes(16)` to generate random candidate messages.

Try this with 8, 12, and 16-bit targets. Do **not** try this with large bit sizes.


In [13]:
def find_preimage(target: int, bits: int, max_attempts: int = 1_000_000) -> tuple[bytes, int] | None:
    """
    Search for a random 16-byte message whose truncated digest equals target.

    Returns (message, attempts) if successful, otherwise None.
    """
    for attempt in range(1, max_attempts + 1):
        candidate = secrets.token_bytes(16)
        if truncated_sha256(candidate, bits) == target:
            return candidate, attempt
    return None


bits = 12
target = secrets.randbelow(2 ** bits)
result = find_preimage(target, bits)
print("target:", target)
print("result:", result)
if result:
    msg, attempts = result
    print("message:", msg.hex())
    print("digest:", truncated_sha256(msg, bits))
    print("attempts:", attempts)


target: 2206
result: (b'\x1c\xf7L\xd3\x121\x04h\xceu,,\xeb\xcc\xda\x81', 601)
message: 1cf74cd312310468ce752c2cebccda81
digest: 2206
attempts: 601


### Reflection

For an n-bit truncated digest, approximately how many attempts would you expect a preimage search to take on average? Did your experiment roughly match that expectation?

Your answer:
On average for a n-bit truncated digest I would expect to get a preimage search in half the total bit. If it is 256 bit I would expect it to be approximately half so 128. It was about what I would expect but the attepts also exceed the actual length and that is becasue the search is looking for a number and the attempts can be looking at the surrounding numbers so the attempts is more.  


## Part 6: Toy Collision Search and the Birthday Effect

Collision search is different from preimage search. Instead of targeting a specific digest, we only need to find two different messages that produce the same digest.

This is much easier because every pair of generated messages creates a collision opportunity.


### Exercise 4

Implement `find_collision()` for a truncated hash.

The function should:

1. Generate random 16-byte messages.
2. Compute the truncated digest.
3. Store the first message seen for each digest.
4. Stop when a new message produces a digest already seen.

Return the two colliding messages, the digest, and the number of attempts.


In [ ]:
def find_collision(bits: int, max_attempts: int = 1_000_000):
    """
    Search for a collision in truncated SHA-256.

    Returns (m1, m2, digest, attempts) if found, otherwise None.
    """
    # TODO: Implement collision search.
    raise NotImplementedError("Complete find_collision()")


# bits = 16
# collision = find_collision(bits)
# print(collision)
# if collision:
#     m1, m2, digest, attempts = collision
#     print("m1:", m1.hex())
#     print("m2:", m2.hex())
#     print("digest:", digest)
#     print("attempts:", attempts)
#     print("check 1:", truncated_sha256(m1, bits))
#     print("check 2:", truncated_sha256(m2, bits))


### Exercise 5

Compare preimage and collision difficulty.

For 16-bit truncated hashes:

- Preimage search should take on the order of 2^16 attempts.
- Collision search should take on the order of 2^8 attempts.

Run several collision trials and compute the average number of attempts.


In [ ]:
def collision_experiment(bits: int = 16, trials: int = 20) -> list[int]:
    """Run several collision searches and return the attempt counts."""
    # TODO: Implement repeated collision trials.
    raise NotImplementedError("Complete collision_experiment()")


# attempts = collision_experiment(16, 20)
# print("attempts:", attempts)
# print("average:", statistics.mean(attempts))
# print("birthday estimate:", 2 ** (16 / 2))


## Part 7: File Integrity

Hashes are often used to verify that a file has not changed.

The following example writes a file, hashes it, modifies one byte, and hashes it again.


In [ ]:
lab_dir = Path("week05_hash_demo")
lab_dir.mkdir(exist_ok=True)
file_path = lab_dir / "message.txt"

file_path.write_text("This is the original file.\n", encoding="utf-8")

def hash_file(path: Path, algorithm: str = "sha256") -> str:
    """Hash a file in chunks and return a hexadecimal digest."""
    if algorithm == "sha256":
        h = hashlib.sha256()
    elif algorithm == "sha3_256":
        h = hashlib.sha3_256()
    elif algorithm == "blake2b":
        h = hashlib.blake2b(digest_size=32)
    else:
        raise ValueError("Unsupported algorithm")

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

before = hash_file(file_path)
file_path.write_text("This is the original file!\n", encoding="utf-8")
after = hash_file(file_path)

print("Before:", before)
print("After: ", after)
print("Same?  ", before == after)


### Exercise 6

Why is hashing a file useful for integrity verification, but not sufficient for authenticity by itself?

Hint: Who computed the hash? How did you receive the expected hash? Could an attacker replace both the file and the hash?

Your answer:



## Part 8: Misuse - Raw Hashes Are Not Authentication

A common mistake is to build an authentication scheme like this:

```text
SHA256(secret || message)
```

This looks reasonable, but raw hashes are not the right tool for message authentication. Merkle-Damgard hashes can be vulnerable to length-extension attacks when used this way.

The correct standard construction is HMAC, which is designed for message authentication.


In [ ]:
secret_key = secrets.token_bytes(32)
message = b"transfer=100&to=alice"

bad_mac = hashlib.sha256(secret_key + message).hexdigest()
good_mac = hmac.new(secret_key, message, hashlib.sha256).hexdigest()

print("Raw secret-prefix hash:", bad_mac)
print("HMAC-SHA256:          ", good_mac)


### Exercise 7

Explain why `SHA256(secret || message)` is not the same as HMAC. You do not need to implement a length-extension attack. Focus on the engineering lesson.

Your answer:



## Part 9: Toolkit Integration

Add a new module to your toolkit:

```text
crypto_toolkit/
    hashes/
        hashing.py
```

Suggested functions:

```python
def sha256_hex(data: bytes) -> str:
    ...

def sha3_256_hex(data: bytes) -> str:
    ...

def blake2b_hex(data: bytes, digest_size: int = 32) -> str:
    ...

def hash_file(path: str, algorithm: str = "sha256") -> str:
    ...

def compare_digest(a: str, b: str) -> bool:
    ...
```

Use `hmac.compare_digest()` for safe digest comparison.


In [ ]:
# Optional: create starter directories for the toolkit if they do not exist.
toolkit_root = Path("crypto_toolkit")
hashes_dir = toolkit_root / "hashes"
hashes_dir.mkdir(parents=True, exist_ok=True)
(hashes_dir / "__init__.py").touch()

starter_path = hashes_dir / "hashing.py"
if not starter_path.exists():
    starter_code = '"""\nModule: crypto_toolkit.hashes.hashing\n\nEducational utilities for working with cryptographic hash functions.\n\nWARNING:\nThese helpers demonstrate correct use of standard hash functions. They do not\nturn raw hashes into authentication or password-storage mechanisms.\n"""\n\nimport hashlib\nimport hmac\nfrom pathlib import Path\n\n\ndef sha256_hex(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha3_256_hex(data: bytes) -> str:\n    return hashlib.sha3_256(data).hexdigest()\n\n\ndef blake2b_hex(data: bytes, digest_size: int = 32) -> str:\n    return hashlib.blake2b(data, digest_size=digest_size).hexdigest()\n\n\ndef hash_file(path: str | Path, algorithm: str = "sha256") -> str:\n    raise NotImplementedError("Complete this function")\n\n\ndef compare_digest(a: str, b: str) -> bool:\n    return hmac.compare_digest(a, b)\n'
    starter_path.write_text(starter_code, encoding="utf-8")

print(f"Starter module available at {starter_path}")


## Part 10: Final Reflection

Return to today's big question:

**How can a short digest represent a large message securely enough that we can use it for integrity, signatures, and identification?**

Write a short response that uses at least three of the following terms:

- preimage resistance
- second-preimage resistance
- collision resistance
- birthday attack
- digest length
- Merkle-Damgard
- sponge construction
- length extension

Your response:
A short digest securely represents a large message by using a one way encryption that compressed data and prevents any tampering through authentication. Iterative hasing using a compression function that transforms an input to a smaller output which is called the Markle-damgard. The preimage resistance ensures that a hash digest can't be worked backwards to find the original message. Collision resistance makes it so that two different messages cannot produce the same digest which can verify the file integretity and signatures. 


## Submission Checklist

Before submitting:

- [ ] All code cells run without errors.
- [ ] All TODO functions are completed.
- [ ] Concept and reflection questions are answered.
- [ ] Toolkit hashing module has been created or updated.
- [ ] You can explain why hashing is not encryption.
- [ ] You can explain why SHA-256 is not appropriate by itself for password storage or message authentication.
